# Release notes

This page lists the main changes in each version of onceover, newest first. The date of a version is the date of its tag in the repository; packages are published to PyPI and crates.io from the tag. Work that is merged but not yet tagged is collected under Unreleased.


## Version 0.1.4 — 9 November 2026

Version 0.1.4 was tagged and published on 9 November 2026. The main changes since 0.1.3 follow.

### Command line and HTML report

- Installing the package also installs an `onceover` command: it profiles a file from the shell and prints the digest by default, or writes the report as JSON or markdown or renders it as HTML. Several files given together are read as a set of tables — the `profile_many` route. `python -m onceover` is equivalent, and a usage page in this book documents the command.
- `report.to_html()` renders a report as one self-contained HTML file: the digest, the findings grouped by category, and a chart for each numeric column, with the charting library embedded so the file opens offline. It needs the `html` extra.

### The book and the benchmark corpus

- The benchmark corpus entry for the satellites table now runs the order-constraints check; the statements it produces were verified against the raw data before the new expectations were recorded. Machine-verified corpus expectations stand at 93 of 93.
- Three methodology pages work one check through on real data: tested associations, including which strong pairs the Simpson check does not examine; values piled up at one point of a column (boundary mass); and order constraints between columns.
- The glossary defines account, deadline, digest and source — four terms the report and this book used without defining.

### Column types

- Decimal columns are analysed as numbers. Previously they were accepted but opaque: counted for rows and nulls, with no statistics and no detector examining them. A decimal column now receives the numeric card, is eligible for the measure role, and is visible to every pass that consumes numeric columns. Values are converted to float at extraction (the stored scaled integer divided by 10^scale), so digits beyond float precision are not retained.


## Version 0.1.3 — 7 November 2026

Version 0.1.3 was tagged and published on 7 November 2026. The main changes since 0.1.2 follow.

### The report

- The report schema moved from 0.2 to 0.7. The table at the end of this page lists each step and what it changed.
- `report.to_digest(chars)` returns a digest of the report that fits a stated character budget.
- `report.diff(other)` reports the findings added, removed or changed between two profiles of a table, matched by stable id, plus coverage status changes.
- `report.to_json(max_chars=...)` returns a size-budgeted JSON view: findings are kept in full in rank order until the budget is reached and collapse to index entries after that, and a `budget` record in the output states what was cut.
- A strong association statement is qualified when the association reverses inside every subgroup of a category column (a Simpson reversal) and the combined within-subgroup effect is significant. The qualification states the per-subgroup numbers; the original statement stays in the report.
- Predictive statements name the columns that drive the prediction, not every input column.
- Statements are checkable against the data: a categorical outlier statement lists every row its share counts, and findings carry the values a reader needs to reproduce them.

### Detectors

- Ordered partitions: regime trees over a time axis or a dimension, split where the level or the trend changes.
- Boundaries: a value piled at a column's observed minimum or maximum, the shape a cap or a floor on recording would leave.
- Missingness: each column missing on at least 1% of rows, the pattern behind the missingness when one is clear, and blocks of columns that go missing on the same rows.
- Content: what a text column actually holds — numbers, dates or booleans stored as text, recognisable formats such as email addresses and URLs, and labels that differ only in case or spacing.
- Order constraints between pairs of columns: one column's values preceding another's in every row, such as a start column before an end column.
- Near-duplicate rows (beyond what chance coincidence on the compared columns explains), values swapped between rows, and rows sitting off a line.
- Most of the rest of the battery also dates from this cycle: breaks in level, trend and variance over time, with transient excursions; associations computed separately per period; long-format (pivot) structure; multi-term arithmetic identities; subset-conditional and categorical surprises; cross-sectional discontinuities; duplicate encodings; interaction strength between predictors; target-aware shape decomposition; digit fingerprints; sentinel values; hotspots; granularity changes in float columns; drift along a time axis; out-of-order appended blocks (backfills).

### Inputs and budgets

- `profile()` accepts a Polars LazyFrame or a file path (Parquet, CSV or IPC) as well as a DataFrame. A large source is profiled from a stratified sample of at most 200,000 rows by default, and the report states whether a number rests on the full data or on the sample, and how large the source is. `exact=True` collects the whole source regardless of the cap, and `sample_seed` fixes the seed the sample is drawn with, so the same source, cap and seed give the same sample and the same report.
- `profile_many()` reads a set of tables together and reports candidate foreign keys: column pairs from different tables whose distinct non-null values are fully contained in a unique column on the other side. Candidates whose column names agree form a strong tier; the rest are reported too, each with a chance score for the containment happening by accident. Partial containment into a unique parent is reported as an overlap with exact orphan counts. The result is a separate `MultiTableReport`; the individual tables are not profiled.
- A caller can declare hints: the time axis, an ordering, gap-free columns, and which columns are dimensions and which are measures.
- Frames with narrow-integer, decimal, struct or fixed-size-array columns are read in instead of failing at ingestion.
- A time budget is also checked inside the five longest passes (the dependence screen, hotspots, discontinuities, surprises and duplicate encodings). A pass stopped mid-way reads `capped` in coverage and states how much of its search it examined. Work before the first check (extraction, column cards, the frame model) cannot be interrupted, so a very short budget can still be exceeded.
- Coverage has an entry for every registered detector. A detector that did not run says why: not enabled, opt-in, quick tier, or deadline reached.
- `profile(timing=True)` records per-pass wall time as `elapsed_ms` on each coverage entry. Findings are identical either way.
- Passes with more candidates than their budget allows choose by a cheap pre-score instead of by column order: ordered partitions, hotspots and the breaks scan.
- Five detectors that never fired on the benchmark corpus (dates, pivot, period associations, drift, granularity) are opt-in rather than default. Report size, not time, is the binding limit for the agents that read these reports.
- The hotspots and dependence searches stream their enumeration, so a wide table does not need its full search space held in memory.
- The discontinuities pass reuses the axis data it computes across targets. On the Taiwanese bankruptcy data the pass took 8,770 ms where it took 13,055 ms before, with identical reports.
- The supported Polars range remains 1.28 and later, as well as 2.x. A wheel test exercises the oldest supported 1.x release; a failure there was traced to a date construction inside one test and fixed in the test, with no change to the supported range.

### The evidence behind the reports

- The benchmark harness stopped counting documented misses as passes. Machine-verified expectations stand at 89 of 89; documented cases are recorded as found (18), partial (4) or missed (10). A planted-signal battery with pinned baselines was added beside it.
- Correctness fixes preserve exact values end to end and bound the work a profile does.
- Functional dependencies are scored and selected by their chance-corrected information (reliable fraction of information), so a determinant with nearly one value per row no longer ranks as a strong dependency. Composite determinants that look unique only on the screening sample are verified on all rows instead of dismissed.
- The terminal-digit test compares a column with the digits its own values can end in (a column of even numbers with the five even digits) and runs only when the central 90% of the values spans at least three cycles of the last digit.
- An appendix to this book evaluates foundation-model screeners (Kumo and Timestral) against plain baselines, with each result stated next to its baseline.


## Version 0.1.2 — 27 October 2026

Report schema 0.2.

- onceover works with Polars 1.28 and later, as well as Polars 2.x. Before this version it required Polars 2.x.


## Version 0.1.1 — 27 October 2026

Report schema 0.2.

- New views detector: a greedy search that groups columns into views, subsets whose joint behaviour is simpler than the whole table's. The detector is opt-in.
- Correctness fixes in the dependence screen and the views search: a marginal distribution, merges that added nothing, selector collisions, and leave-one-out assignment. Budget use is reported more clearly.


## Version 0.1.0 — 26 October 2026

First release. Report schema starts at 0.2.

- A profile of a Polars DataFrame returns column summaries, relationship findings with the evidence beside each claim, and a record of what was searched (coverage).


## Report schema history

Every tagged version before 0.1.3 reports schema 0.2. Each later step is in version 0.1.3.

| Report schema | Version | What changed in the report |
| --- | --- | --- |
| 0.2 | 0.1.0 | Evidence-first report: findings carry their evidence beside the claim. Per-type evidence payloads were added within 0.2. |
| 0.3 | 0.1.3 | Every finding carries facets — scope, object, evidence, conditionality, temporality and status as fields — and its category is derived from the facets by a fixed table. |
| 0.4 | 0.1.3 | The frame model: each column's semantic role and the assembled time axis are computed before the detectors run and included in the report. |
| 0.5 | 0.1.3 | An account block states the table's grain, key and time axis. The digest re-ranks statements and demotes restatements. |
| 0.6 | 0.1.3 | Verified relationship families collapse to one lead statement; the remaining members are listed in `grouped_findings`. |
| 0.7 | 0.1.3 | Associations are tested pair by pair: the report lists every tested pair with its q-value, weak pairs collapse into one digest line, and `report.associations()` returns the full table. A Simpson-reversal qualification for strong statements was added within 0.7. |
